# Day 47 — PyTorch basics: tensors, autograd, simple NN
Objectives:
- Build an MLP.
- Training/validation split.
- Evaluate accuracy.
Using sklearn make_classification for simplicity.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
X,y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
sc = StandardScaler(); X = sc.fit_transform(X).astype(np.float32)
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
Xtr, Xte = torch.from_numpy(Xtr), torch.from_numpy(Xte)
ytr, yte = torch.from_numpy(ytr).long(), torch.from_numpy(yte).long()
model = nn.Sequential(nn.Linear(20,64), nn.ReLU(), nn.Linear(64,2))
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(50):
    opt.zero_grad(); out = model(Xtr); loss = loss_fn(out, ytr); loss.backward(); opt.step()
pred = model(Xte).argmax(1); acc = (pred==yte).float().mean().item(); acc


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — logits, minibatches, modes, and a correct PyTorch training loop

### Mental model

`nn.Module` composes registered parameters and operations. For
multiclass classification, the final layer returns one **logit** per
class; `CrossEntropyLoss` combines log-softmax and negative
log-likelihood, so the model should not apply softmax first. Targets are
integer class indices with `torch.long` dtype.

A `DataLoader` defines batch and shuffle behavior. Training mode enables
dropout and batch-normalization updates; evaluation mode disables those
training behaviors but does not itself disable gradients. Use
`model.eval()` together with `torch.no_grad()` for validation.

### Read the API before running it

- **`DataLoader(dataset, batch_size=..., shuffle=...)`:** yields bounded feature/target batches and controls training order.
- **`model(features)`:** returns logits shaped `(batch, classes)` for multiclass classification.
- **`CrossEntropyLoss(logits, labels)`:** expects floating logits and `long` class indices shaped `(batch,)`.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — make the logits-and-label contract executable

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Class labels are zero-based indices and no softmax was applied before the loss.

In [ ]:
import torch
from torch import nn

model = nn.Linear(3, 2)
features = torch.tensor([[1.0, 0.0, -1.0], [0.5, 2.0, 1.0]])
labels = torch.tensor([0, 1], dtype=torch.long)
logits = model(features)
loss = nn.CrossEntropyLoss()(logits, labels)
print({"logit_shape": tuple(logits.shape),
       "label_shape": tuple(labels.shape), "loss": loss.item()})
assert logits.shape == (2, 2) and loss.ndim == 0

**Expected observation:** Two rows and two classes produce a `(2, 2)` logit tensor and one scalar batch loss.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — observe dropout differ between train and eval modes

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** A fixed seed controls the demonstration; production quality is not inferred from one mask.

In [ ]:
import torch
from torch import nn

torch.manual_seed(4702)
layer = nn.Dropout(p=0.5)
values = torch.ones(12)
layer.train()
train_output = layer(values)
layer.eval()
with torch.no_grad():
    eval_output = layer(values)
print({"train_zeros": int((train_output == 0).sum()),
       "eval_unchanged": bool(torch.equal(eval_output, values))})
assert torch.equal(eval_output, values)

**Expected observation:** Dropout randomly masks/scales values in training but becomes an identity operation in evaluation.

### Debugging and practice ramp

**Common mistake:** Passing probabilities to `CrossEntropyLoss`, float labels, or evaluating while the model remains in training mode.

**Diagnostic:** Assert every batch's shape/dtype/device, inspect one loss and gradient norm, and compare repeated predictions in train versus eval mode.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define logits, minibatches, modes, and a correct PyTorch training loop in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not add schedulers, early stopping, or mixed precision until the basic loop overfits one tiny batch and validates correctly.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Add dropout to the MLP and compare results.

**Verify:** For task `Add dropout to the MLP and compare results`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






2. Implement a small minibatch training loop with `DataLoader`.

**Verify:** For task `Implement a small minibatch training loop with DataLoader`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







### Progressive hints

1. Place `Dropout(p=...)` after an activation. Compare multiple seeded runs or
   validation curves, because one final accuracy is noisy.
2. Wrap float32 features and long labels in a `TensorDataset`; shuffle only the
   training loader. Move `zero_grad`, forward, loss, backward, and step inside
   the batch loop.

The separate solution proceeds to early stopping, `StepLR`, and CUDA mixed
precision. Mixed precision is an optional GPU optimization; the code path must
remain correct on CPU.

### Additional mastery practice

Build a training loop whose dtype, shape, mode, randomness, and checkpoint state can be inspected and resumed on CPU without hidden notebook state.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Shape and dtype contract:** Write assertions at the start of a classification training step for feature shape/dtype, target shape/dtype, and logits shape.
   **Progressive hint:** CrossEntropyLoss expects floating logits `(batch, classes)` and integer class indices `(batch,)` with dtype long.

**Verify:** For task `Shape and dtype contract: Write assertions at the start of a classification training step for...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







4. **Validation implementation:** Implement an evaluation function that returns sample-weighted loss and accuracy, restores the caller's prior train/eval mode, and never retains an autograd graph.
   **Progressive hint:** Remember `was_training = model.training`, call eval and no_grad, aggregate counts, then restore train mode only if it was previously active.

**Verify:** For task `Validation implementation: Implement an evaluation function that returns sample-weighted loss...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







5. **DataLoader reproducibility:** Run two shuffled DataLoaders with the same seed and compare batch order. Then state what changes when using worker processes.
   **Progressive hint:** Pass a seeded `torch.Generator`; worker initialization and external NumPy/Python randomness need their own deliberate seeds.

**Verify:** For task `DataLoader reproducibility: Run two shuffled DataLoaders with the same seed and compare batch...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.







6. **Portable checkpoint:** Save model, optimizer, epoch, metric history, and configuration, then reload on CPU and resume one step. Explain `state_dict` versus serializing the entire model object.
   **Progressive hint:** Save plain state dictionaries plus architecture/config metadata. Use `map_location='cpu'` and recreate the model class before loading.

**Verify:** For task `Portable checkpoint: Save model, optimizer, epoch, metric history, and configuration, then re...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Shape and dtype contract


# Practice 4 — Validation implementation


# Practice 5 — DataLoader reproducibility


# Practice 6 — Portable checkpoint
